In [0]:
# Notebook 01 - Camada Bronze com tratamento do separador ';'
from pyspark.sql.functions import col, split, trim, current_timestamp

# 1. Lê a tabela carregada no Unity Catalog
df_raw = spark.read.table("workspace.default.base_chegadas_2026_acum_08_ago")

# Identifica o nome da coluna única original
coluna_bruta = df_raw.columns[0]

# 2. Separa a coluna única dividida por ';' nas colunas corretas
df_bronze = df_raw.withColumn("campos", split(col(coluna_bruta), ";")) \
    .select(
        trim(col("campos")[0]).alias("via_acesso"),
        trim(col("campos")[1]).alias("uf_destino"),
        trim(col("campos")[2]).alias("nome_pais_correto"),
        trim(col("campos")[3]).alias("mes"),
        trim(col("campos")[4]).alias("ano"),
        trim(col("campos")[5]).alias("chegadas"),
        current_timestamp().alias("data_ingestao")
    ) \
    .filter(col("via_acesso") != "Via_de_acesso") # Remove a linha do cabeçalho original

# 3. Salva a tabela na Camada Bronze em formato Delta Lake
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_chegadas_turistas")

print("Tabela Bronze (workspace.default.bronze_chegadas_turistas) gerada com sucesso!")

---------------------------------------------------------------------------
ArrayIndexOutOfBoundsException            Traceback (most recent call last)
File <command-8699350401734222>, line 27
     11 df_bronze = df_raw.withColumn("campos", split(col(coluna_bruta), ";")) \
     12     .select(
     13         trim(col("campos")[0]).alias("via_acesso"),
   (...)     20     ) \
     21     .filter(col("via_acesso") != "Via_de_acesso") # Remove a linha do cabeçalho original
     23 # 3. Salva a tabela na Camada Bronze em formato Delta Lake
     24 df_bronze.write \
     25     .format("delta") \
     26     .mode("overwrite") \
---> 27     .saveAsTable("workspace.default.bronze_chegadas_turistas")
     29 print("Tabela Bronze (workspace.default.bronze_chegadas_turistas) gerada com sucesso!")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:828, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    826 self._write.table_

In [0]:
# Notebook 01 - Camada Bronze com tratamento seguro usando get()
from pyspark.sql.functions import col, split, trim, current_timestamp, get

# 1. Lê a tabela carregada no Unity Catalog
df_raw = spark.read.table("workspace.default.base_chegadas_2026_acum_08_ago")

# Identifica o nome da coluna única
coluna_bruta = df_raw.columns[0]

# 2. Divide a linha em array
df_split = df_raw.withColumn("campos", split(col(coluna_bruta), ";"))

# 3. Extrai com segurança cada campo usando get() para evitar erro de índice
df_bronze = df_split.select(
    trim(get(col("campos"), 0)).alias("via_acesso"),
    trim(get(col("campos"), 1)).alias("uf_destino"),
    trim(get(col("campos"), 2)).alias("nome_pais_correto"),
    trim(get(col("campos"), 3)).alias("mes"),
    trim(get(col("campos"), 4)).alias("ano"),
    trim(get(col("campos"), 5)).alias("chegadas"),
    current_timestamp().alias("data_ingestao")
).filter(
    (col("via_acesso") != "Via_de_acesso") & 
    (col("via_acesso").isNotNull()) & 
    (col("nome_pais_correto").isNotNull())
)

# 4. Salva a tabela na Camada Bronze em formato Delta Lake
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_chegadas_turistas")

print("Tabela Bronze (workspace.default.bronze_chegadas_turistas) gerada com sucesso!")

Tabela Bronze (workspace.default.bronze_chegadas_turistas) gerada com sucesso!


In [0]:
# Notebook 02: Camada Silver (Limpeza, Correção de Acentos e Filtro do Mercado Espanhol)
from pyspark.sql.functions import col, upper, trim, regexp_replace

# 1. Lê a tabela da Camada Bronze
df_bronze = spark.read.table("workspace.default.bronze_chegadas_turistas")

# 2. Tratamento de caracteres especiais/acentos e padronização
df_silver_clean = df_bronze \
    .filter(trim(col("nome_pais_correto")) == "Espanha") \
    .withColumn("via_acesso_clean", trim(col("via_acesso"))) \
    .withColumn("via_acesso_clean", regexp_replace(col("via_acesso_clean"), "A.rea", "Aérea")) \
    .withColumn("via_acesso_clean", regexp_replace(col("via_acesso_clean"), "Area", "Aérea")) \
    .withColumn("uf_clean", upper(trim(col("uf_destino")))) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "S.O PAULO|SO PAULO", "SÃO PAULO")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "PARAN.|PARAN", "PARANÁ")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "CEAR.|CEAR", "CEARÁ")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "MARANH.O|MARANHO", "MARANHÃO")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "PARA.BA|PARABA", "PARAÍBA")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "ROND.NIA|RONDNIA", "RONDÔNIA")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "AMAP.|AMAP", "AMAPÁ")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "GOI.S|GOIS", "GOIÁS")) \
    .withColumn("uf_clean", regexp_replace(col("uf_clean"), "PIAU.|PIAU", "PIAUÍ"))

# 3. Seleção final com os campos corrigidos e tipagem de dados
df_silver = df_silver_clean.select(
    col("via_acesso_clean").alias("via_acesso"),
    col("uf_clean").alias("uf_destino"),
    col("nome_pais_correto").alias("pais_origem"),
    col("mes").alias("mes_chegada"),
    col("ano").cast("int").alias("ano_chegada"),
    col("chegadas").cast("int").alias("quantidade_turistas"),
    col("data_ingestao")
).filter(col("quantidade_turistas").isNotNull())

# 4. Grava a tabela corrigida na Camada Silver no Delta Lake
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_chegadas_espanha")

print("Tabela Silver (workspace.default.silver_chegadas_espanha) atualizada e corrigida com sucesso!")

Tabela Silver (workspace.default.silver_chegadas_espanha) atualizada e corrigida com sucesso!


In [0]:
# Notebook 02: Camada Silver - Correção do caractere corrompido \uFFFD
from pyspark.sql.functions import col, upper, trim, regexp_replace, replace

# 1. Lê a Camada Bronze
df_bronze = spark.read.table("workspace.default.bronze_chegadas_turistas")

# 2. Corrige especificamente o caractere invalido \uFFFD (losango com interrogação)
df_silver_clean = df_bronze \
    .filter(trim(col("nome_pais_correto")) == "Espanha") \
    .withColumn("via_acesso_corr", regexp_replace(col("via_acesso"), "\uFFFD", "é")) \
    .withColumn("uf_corr", upper(trim(col("uf_destino")))) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "\uFFFD", "")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "SO PAULO", "SÃO PAULO")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "PARAN", "PARANÁ")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "CEAR", "CEARÁ")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "MARANHO", "MARANHÃO")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "PARABA", "PARAÍBA")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "RONDNIA", "RONDÔNIA")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "AMAP", "AMAPÁ")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "GOIS", "GOIÁS")) \
    .withColumn("uf_corr", regexp_replace(col("uf_corr"), "PIAU", "PIAUÍ"))

# 3. Seleção e tipagem final
df_silver = df_silver_clean.select(
    trim(col("via_acesso_corr")).alias("via_acesso"),
    trim(col("uf_corr")).alias("uf_destino"),
    col("nome_pais_correto").alias("pais_origem"),
    col("mes").alias("mes_chegada"),
    col("ano").cast("int").alias("ano_chegada"),
    col("chegadas").cast("int").alias("quantidade_turistas"),
    col("data_ingestao")
).filter(col("quantidade_turistas").isNotNull())

# 4. Sobrescreve a tabela Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_chegadas_espanha")

print("Camada Silver corrigida com sucesso!")

Camada Silver corrigida com sucesso!


In [0]:
%sql
-- Criação da Camada Gold com agregação de turistas espanhóis por estado brasileiro e via de acesso
CREATE OR REPLACE TABLE workspace.default.gold_turistas_espanha_por_uf AS
SELECT 
    uf_destino,
    via_acesso,
    SUM(quantidade_turistas) AS total_turistas_espanhois,
    MAX(data_ingestao) AS ultima_atualizacao
FROM 
    workspace.default.silver_chegadas_espanha
GROUP BY 
    uf_destino,
    via_acesso
ORDER BY 
    total_turistas_espanhois DESC;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM workspace.default.gold_turistas_espanha_por_uf LIMIT 10;

uf_destino,via_acesso,total_turistas_espanhois,ultima_atualizacao
SÃO PAULO,Aérea,60378,2026-09-27T23:49:13.998Z
RIO DE JANEIRO,Aérea,23789,2026-09-27T23:49:13.998Z
PARANÁ,Terrestre,10449,2026-09-27T23:49:13.998Z
CEARÁ,Aérea,6194,2026-09-27T23:49:13.998Z
PERNAMBUCO,Aérea,4930,2026-09-27T23:49:13.998Z
BAHIA,Aérea,4223,2026-09-27T23:49:13.998Z
DISTRITO FEDERAL,Aérea,1161,2026-09-27T23:49:13.998Z
RIO GRANDE DO NORTE,Aérea,964,2026-09-27T23:49:13.998Z
SANTA CATARINA,Aérea,946,2026-09-27T23:49:13.998Z
MINAS GERAIS,Aérea,604,2026-09-27T23:49:13.998Z
